# Analysis of SAR Data #

This notebook is meant to catalogue some basic methods of SAR data analysis. As a use case, this notebook and the accompanying functions will focus on vessel detection using both scientific multi-channel SAR raster data and pixelized SAR images. 

The purpose of this notebook is to serve as a personal record of SAR image processing, but I plan to publish it on Github as a public journal. 

In [ ]:
%load_ext autoreload
%autoreload 2

### Multichannel Raster Data ### 

This first cell walks through some basic data ingestion for scientific SAR data. This cell pulls data from Sentinel-1 through Planetary Computer's STAC API, it then performs basic analysis on the channels that are available and what they look like. 

In [ ]:
# Logical Dependencies
import pystac_client
import planetary_computer

# Local Dependencies
from config import PLANETARY_COMPUTER_SENTINEL1
from sar_multichannel_analysis import multichannel_data_analysis


# Setup Planetary Computer connection
catalog = pystac_client.Client.open(PLANETARY_COMPUTER_SENTINEL1)

search = catalog.search(
    collections=["sentinel-1-rtc"],
    bbox=[-5.8, 35.7, -5.1, 36.3],
    datetime="2025-01-01/2025-12-31",
    max_items=10,
)

# Item Analysis
items = list(search.items())
multichannel_data_analysis(items)

### SSDD Dataset Analysis ### 

This section walks through the basic composition and analysis of the SSDD dataset. This Dataset is prepartitioned by huggingface into train, test, validate sets. 

It is worth noting that this dataset contains static SAR images rather than raw polarized data. As a result this data has already been abstracted in post-processing. More detailed analysis may be performed with the polarized data, but I do not currently have labels for a dataset like that. 

This first cell creates a data loader object to help streamline the data handling. It then shows a tree of the data within.

In [ ]:
# To download the data, uncommend the following two lines and run this cell
#from helpers.download_ssdd import download_ssdd
#download_ssdd()

# Dependencies
from helpers.data_loader import DataLoader

# If you are pulling the SSDD data from elsewhere, edit the line below
from config import BASE_PATH
SSDD_PATH = BASE_PATH

# Setup Dataloader
dl = DataLoader(SSDD_PATH)



A sample of the data pairings can be extracted and displayed with the cell below.

In [ ]:
# Get all the image paths for training images
train_files = dl.get_images()

# Consider and display the content for the combined sample of one of these images
sample = dl.load_sample(train_files[0])
sample.show()

## Ship Identification ##

As a data processing use case, the next section of this notebook will attempt to use the labelled SSDD data to predict the location of ships in an ocean scene. Both Determinitsic and Deep Learning approaches will be used and then compaired. 


### Deterministic Prediction ### 

The first method of prediciton will be to use the scene's brightness contrast to identify areas of high energy return. This approach assumes that in an oceanic scene, all areas of high contrast (with a minimum pixel size filter) will represent a ship. This is a fairly safe assumption for this level of analysis, but it is worth noting that there may be cases of false positives. 

The first attempt at this will be simple and will include:
1. Establish an average energy for the scene by determining the average gray value
1. Call out all pixels in the scene with brightness values above one standard deviation of this average
1. Consider all collections of more than a base number of pixels (e.g. 5 pixels)

In [ ]:
# Load the deterministic model
from deterministic_model import mean_threshold_model

# Execute the sample
sample = dl.load_sample(dl.get_images()[0])
pred = mean_threshold_model(sample, show_checkpoints=True)

# Evaluate the prediction
results = sample.evaluate(pred, iou_threshold=0.3, plot=True)

With one sample complete, it is useful to test the deterministic model on the validation set to determine overall performance.

In [ ]:
import pandas as pd
import numpy as np

dl.mode = 'valid'
img_list = dl.get_images()

records = []

# Iterate through all the images, evaluate and record performance
for img in img_list:
    # Evaluate
    sample = dl.load_sample(img)
    pred = mean_threshold_model(sample)
    results = sample.evaluate(pred, iou_threshold=0.3)

    # Record results
    new_row = results.copy()
    del(new_row['matches'])
    del(new_row['iou_matrix'])
    del(new_row['false_positive_indices'])
    del(new_row['false_negative_indices'])

    records.append(new_row)  


# Generate Composite Results
all_results = pd.DataFrame(records)
vdf = dl.inventory[dl.inventory['split']=='valid']
total_ships = vdf['n_ships'].sum()
calc_results = {
    'True_Positives_Detected': np.sum(all_results.true_positives),
    'True_Positive_Percentage': np.sum(all_results.true_positives)/total_ships,
    'False_Positive_Detection': np.sum(all_results.false_positives)/total_ships,
    'False_Negative_Detection': np.sum(all_results.false_negatives)/total_ships,
    'Average_Precision': np.average(all_results.precision),
    'Average_Recall': np.average(all_results.recall),
    'Average_F1': np.average(all_results.f1)
}

print('Results summary over Validation Set: \n')
for key, value in calc_results.items(): print(f'   - {key}: {value}')


In [ ]:
incorrect = all_results[all_results['recall']<0.5]
limit = 7
i = 0
for img in incorrect['path']:
    i+=1
    if i > limit: break
    sample = dl.load_sample(img)
    pred = mean_threshold_model(sample)
    sample.evaluate(pred,iou_threshold=0.3, plot=True)

**Analysis:**

Note that in these reviews that in many cases, the images that were inaccurate fell under two cases:
1. Shore Images (False Negatives) - Images that were taken while the ship was close to port resulted in poor accuracy because the shoreline raised the mean pixel value of the scene such that ships were not evenly distinguished
1. Wakes (False Positives) - In many cases, the wake trailing the ships created areas of high reflectivity, this is likely becuase Bragg resonance was achieved as the ship altered the water's wave pattern

## Deep Learning with YOLO ##  

Now we I will perform a similar analysis on the images using the training and testing sets to fine-tune a small YOLO model. The model will then be used to evaluate the validation images and results will be compared against the deterministic model.

Base model selected: YOLO1 nano *(yolo11n.pt)*

This training will be conducted using Mac MPS

In [ ]:
#%pip install ultralytics pyyaml
import yaml
import torch
import ultralytics
from pathlib import Path
from config import BASE_PATH

# Set the Device

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("Apple GPU available:", torch.backends.mps.is_available())
print("CUDA GPU available:", torch.cuda.is_available())

if torch.cuda.is_available(): device = 0 # NVIDIA GPU
elif torch.backends.mps.is_available(): device = "mps" # Your M1 GPU
else: device = "cpu"

print("Training device:", device, '\n')

# Setup the Config

DATA_ROOT = Path(BASE_PATH).resolve()
CONFIG_PATH = Path("ssdd.yaml").resolve()

config = {
    "path": str(DATA_ROOT),
    "train": "images/train",
    "val": "images/valid",
    "test": "images/test",
    "names": {0: "ship"},
}

CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False))
print('Config:')
print(CONFIG_PATH.read_text())

In [ ]:
# Download pretrained weights
model = ultralytics.YOLO("yolo11n.pt")  

model.train(
    data=str(CONFIG_PATH),
    epochs=2,
    imgsz=640,
    batch=4,
    device=device,
    workers=0,
    project="runs/ssdd",
    name="setup_check",
)
